# PCU-SPARSE-PATH-DEPTH-001 — Dual-GPU 3/4/5-layer sparse path sweep

Engineering-only depth kill test. Every topology exactly replays and freezes the published L7/K64 hybrid association state. It then adds a fixed K32 / 256-step transport+readout budget distributed across 3, 4, or 5 nested layers. Depth-3 and depth-4 run concurrently on two isolated CUDA processes; depth-5 follows on GPU0. Formal seeds are never executed.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys

BRANCH = 'codex/pcu-composability-kill-001'
REPO = Path('/kaggle/working/mini-cells')
OUT = REPO / 'artifacts/research/pcu-sparse-path-depth-001/engineering/26090501-depth3-4-5'
PREVIOUS = REPO / 'artifacts/research/pcu-cross-layer-readout-001/engineering/26090501-l7k64-plus-l23k16'
FORMAL_SEEDS = (26090511, 26090512, 26090513)
REQUIRED_TRANSFORMERS = '5.16.1'
os.environ.setdefault('HF_HOME', '/kaggle/working/hf-cache')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

def run(cmd, *, env=None, capture=False):
    cmd = [str(x) for x in cmd]
    print('+', ' '.join(cmd))
    result = subprocess.run(cmd, check=True, env=env, text=True, capture_output=capture)
    return result.stdout.strip() if capture else ''

if not REPO.exists():
    run(['git', 'clone', '--branch', BRANCH, 'https://github.com/ArcheLabs/mini-cells.git', REPO])
os.chdir(REPO)
run(['git', 'fetch', 'origin'])
run(['git', 'checkout', BRANCH])
run(['git', 'pull', '--ff-only', 'origin', BRANCH])
run([sys.executable, '-m', 'pip', 'install', '-e', '.[dev]'])
run([sys.executable, '-m', 'pip', 'install', f'transformers=={REQUIRED_TRANSFORMERS}', 'huggingface_hub>=0.36,<2.0', 'safetensors>=0.4', 'accelerate>=1.0'])

import torch, transformers
assert transformers.__version__ == REQUIRED_TRANSFORMERS
assert torch.cuda.is_available()
assert torch.cuda.device_count() >= 2, f'This experiment requires two GPUs, got {torch.cuda.device_count()}'
print(json.dumps({
    'commit': run(['git', 'rev-parse', 'HEAD'], capture=True),
    'tree': run(['git', 'rev-parse', 'HEAD^{tree}'], capture=True),
    'gpu0': torch.cuda.get_device_name(0),
    'gpu1': torch.cuda.get_device_name(1),
    'transformers': transformers.__version__,
}, indent=2))


In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
secrets = UserSecretsClient()
hf_token = secrets.get_secret('HF_TOKEN')
github_token = secrets.get_secret('GITHUB_TOKEN')
assert hf_token and github_token
os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
os.environ['GITHUB_TOKEN'] = github_token
login(token=hf_token, add_to_git_credential=False)
print('Secrets loaded; token values were not printed.')


In [ ]:
SEED_REGISTRY = REPO / 'research/formal_seed_registry.json'
def formal_states():
    payload = json.loads(SEED_REGISTRY.read_text())
    return {int(row['seed']): row['state'] for row in payload['seeds']}
expected = {seed: 'RESERVED_UNTOUCHED' for seed in FORMAL_SEEDS}
assert formal_states() == expected
assert run(['git', 'hash-object', SEED_REGISTRY], capture=True) == '71a3015a7d54e795538b3aa6750860f0b9168cb3'
previous = json.loads((PREVIOUS / 'DECISION.json').read_text())
assert previous['status'] == 'CROSS_LAYER_READOUT_IMPROVES_BUT_DOES_NOT_RESCUE'
assert abs(previous['cross_layer_direct_accuracy'] - 0.15625) < 1e-12
assert abs(previous['cross_layer_ranking_accuracy'] - 0.828125) < 1e-12
remote_path = 'artifacts/research/pcu-cross-layer-readout-001/engineering/26090501-l7k64-plus-l23k16/DECISION.json'
assert subprocess.run(['git', 'show', f'origin/{BRANCH}:{remote_path}'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0
print(json.dumps({'formal_seed_states': formal_states(), 'previous_cross_layer': previous['status']}, indent=2))


In [ ]:
test_env = os.environ.copy()
test_env['PYTHONPATH'] = str(REPO / 'src')
test_env['PYTEST_DISABLE_PLUGIN_AUTOLOAD'] = '1'
run([sys.executable, '-m', 'pytest', '-q', 'tests/research/05-pcu-kill-001'], env=test_env)
run([sys.executable, '-m', 'compileall', '-q', 'src/minicells/pcu_kill_001', 'scripts/research'])
print('PCU sparse-path depth test/compile gate: PASS')


In [ ]:
if OUT.exists():
    existing = sorted(p.name for p in OUT.glob('*.json'))
    assert not existing, f'Sparse-path output already exists; inspect before rerun: {existing}'
run([
    sys.executable, 'scripts/research/run_pcu_sparse_path_depth_001.py',
    '--seed', '26090501',
    '--out', OUT,
])
assert formal_states() == expected


In [ ]:
required = ['RUN_IDENTITY.json', 'DESIGN.json', 'RESULT.json', 'DECISION.json', 'DEPTH_3.json', 'DEPTH_4.json', 'DEPTH_5.json']
missing = [name for name in required if not (OUT / name).is_file()]
assert not missing, missing
identity = json.loads((OUT / 'RUN_IDENTITY.json').read_text())
decision = json.loads((OUT / 'DECISION.json').read_text())
result = json.loads((OUT / 'RESULT.json').read_text())
assert identity['dual_gpu_execution'] is True and identity['gpu_count'] == 2
assert decision['valid_run'] is True and decision['formal_execution_not_started'] is True
assert decision['nested_topologies'] is True
assert decision['total_added_k_each'] == 32
assert decision['total_added_steps_each'] == 256
assert formal_states() == expected
print(json.dumps({
    'status': decision['status'],
    'best_depth': decision['best_depth'],
    'best_direct_accuracy': decision['best_direct_accuracy'],
    'depths': decision['depths'],
    'dual_gpu_execution': identity['dual_gpu_execution'],
    'formal_seed_states': formal_states(),
}, indent=2))


In [ ]:
run([sys.executable, 'scripts/research/publish_pcu_sparse_path_depth_001.py', '--branch', BRANCH])
assert formal_states() == expected
print(json.dumps({'published': True, 'formal_seed_states': formal_states()}, indent=2))
